<a href="https://colab.research.google.com/github/jmpellech/BUS-stop/blob/main/BUS_stop_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
#clone our GitHub fork and move into the directory containing the BUS-stop scripts
!git clone https://github.com/jmpellech/BUS-stop.git
%cd BUS-stop/bus-stop-keras

Cloning into 'BUS-stop'...
remote: Enumerating objects: 258, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 258 (delta 23), reused 16 (delta 11), pack-reused 214 (from 1)
Receiving objects: 100% (258/258), 38.62 MiB | 19.13 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/content/BUS-stop/bus-stop-keras/BUS-stop/bus-stop-keras/BUS-stop/bus-stop-keras


In [35]:
#install the libraries needed by the original TensorFlow-based implementation
#use a binary Transformers release compatible with Colab's Python version

!pip install -q "tf-keras==2.21.0" scikit-learn pandas scipy sentencepiece
!pip install -q --only-binary=:all: "transformers==4.51.1"

In [36]:
#use legacy Keras for compatibility with the BUS-stop TensorFlow model code

import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import sys
import tensorflow as tf
import transformers

from transformers import TFBertModel, BertTokenizer, BertConfig

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("BERT imports successful!")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
TensorFlow: 2.21.0
Transformers: 4.51.1
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
BERT imports successful!


In [37]:
#download the pretrained BERT encoder and save its TensorFlow weights locally
#BUS-stop will load this checkpoint when training the sentiment classifier

from transformers import TFBertModel, BertTokenizer

model = TFBertModel.from_pretrained("bert-base-uncased")

model.save_pretrained("./params/bert_tf")

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
tokenizer.save_pretrained("./params/bert_tf")

print("BERT downloaded and saved successfully!")

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

BERT downloaded and saved successfully!


In [39]:
#generate the experimental SST-2 train/unlabeled/test splits.

!python setup_experiments.py

done


In [40]:
#export the Keras compatibility setting for Python launched with !python
%env TF_USE_LEGACY_KERAS=1

env: TF_USE_LEGACY_KERAS=1


In [41]:
#short compatibility test: two epochs and two preliminary BUS-stop runs
!python main.py \
  --task SST-2 \
  --data_path bal/0 \
  --pt_model_checkpoint ./params/bert_tf/ \
  --epochs 2 \
  --T 2 \
  --n_base 1 \
  --batch_size 8 \
  --verbose 1 \
  --log_prefix colab_test_

W0000 00:00:1794279998.197345   22193 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
03:06:38:BUS-stop:INFO: ***Logging start***
03:06:38:BUS-stop:INFO: task_path = ./data/SST-2/bal/0
03:06:38:BUS-stop:INFO: {'task': 'SST-2', 'data_path': 'bal/0', 'seed': 0, 'pt_model': 'TFBertModel', 'pt_model_checkpoint': './params/bert_tf/', 'max_seq_length': 64, 'word_freeze': False, 'learning_rate': 3e-05, 'drop_rate': 0.2, 'epochs': 2, 'batch_size': 8, 'patience': 10, 'val_ratio': 0.5, 'T': 2, 'n_base': 1, 'n_que': 5, 'verbose': 1, 'cali_acc_or_f1': 'f1', 'bias_lab_or_val': 'val', 'log_prefix': 'colab_test_'}
03:06:38:BUS-stop:INFO: Running on /GPU:0
03:06:38:BUS-stop:INFO: devices = ['/GPU:0']
03:06:38:BUS-stop:INFO: Number of devices: 1
03:06:38:BUS-stop:INFO: *******************
03:06:38:BUS-stop:INFO: ***Preprocessing***
03:06:38:BUS-stop:INFO: *******************
03:06:38:BUS-stop:INFO